# Fase 3: Data Preparation (Avanzada / Datos Reales)
En esta fase vamos a conectarnos a APIs para descargar datos **macroeconómicos globales reales** y añadir eventos puntuales descubiertos en el 'Análisis Comprehensivo 2018-2024'. Luego los cruzaremos con los datos de turismo de Guanacaste.

In [ ]:
import pandas as pd
import numpy as np
import os
import datetime

print('Librerías importadas exitosamente.')

### 1. Ingest: Descarga de Variables Globales (Reserva Federal - FRED)

In [ ]:
start_date = '2018-01-01'

series = {
    'UNRATE': 'tasa_desempleo_usa',                 
    'USINFO': 'empleo_sector_tech_usa',             
    'UMCSENT': 'sentimiento_consumidor_michi',      
    'CPIAUCSL': 'inflacion_usa_cpi',                
    'DCOILWTICO': 'precio_petroleo_wti',            
    'DSPIC96': 'ingreso_disponible_usa'             
}

print('Descargando datos macro desde FRED mediante acceso directo a CSV...')
df_list = []

for s_id, s_name in series.items():
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={s_id}'
    df_temp = pd.read_csv(url, parse_dates=['observation_date'], na_values='.')
    df_temp = df_temp.rename(columns={'observation_date': 'DATE'})
    
    df_temp = df_temp[df_temp['DATE'] >= start_date].copy()
    df_temp[s_id] = pd.to_numeric(df_temp[s_id], errors='coerce')
    
    df_temp.set_index('DATE', inplace=True)
    df_temp = df_temp.resample('ME').mean()  
    df_temp.rename(columns={s_id: s_name}, inplace=True)
    
    df_list.append(df_temp)

df_macro = pd.concat(df_list, axis=1)
df_macro.reset_index(inplace=True)
df_macro['DATE'] = df_macro['DATE'].dt.strftime('%Y-%m-%d')

print(f'Descarga completa. Total de meses descargados: {df_macro.shape[0]}')

### 2. Feature Engineering: INTEGRACIÓN DEL ANÁLISIS COMPREHENSIVO TOTAL (2018-2024)
Mapeo de la Tabla de Impacto (Parte 6 del informe: COVID, Huelgas, Canadá, Incendios, Huracanes, Liberia, Marea Roja, Crisis Texas).

In [ ]:
### A. MACRO-EVENTOS GLOBALES / MERCADOS EMISORES

# 1. COVID-19 Agudo (Cierres totales Marzo 2020 - Dic 2021)
df_macro['covid_agudo_cr_mundial'] = df_macro['DATE'].apply(lambda x: 1 if '2020-03-01' <= x <= '2021-12-31' else 0)

# 2. Guerra Ucrania y Crisis Energética Europea (Dispara Jet Fuel + Europa en contracción)
df_macro['guerra_ucrania_activa'] = df_macro['DATE'].apply(lambda x: 1 if x >= '2022-02-01' else 0)

# 3. Inflación Energética Global Post-Pandemia (2021 - 2023)
df_macro['crisis_precios_energia_global'] = df_macro['DATE'].apply(lambda x: 1 if '2021-01-01' <= x <= '2023-12-31' else 0)

# 4. Recesión y Caída GDP per Cápita Canadá
df_macro['stress_economico_canada'] = df_macro['DATE'].apply(lambda x: 1 if x >= '2022-06-01' else 0)

# 5. USA Election Seasons
df_macro['usa_election_season'] = df_macro['DATE'].apply(lambda x: 1 if (('2020-11' <= x <= '2021-01') or ('2024-11' <= x <= '2025-01')) else 0)


### B. SHOCKS REGIONALES CLIMÁTICOS (ESTADOS TOP USA)

# 6. Incendios California (Ago-Nov en 2018, 2020, 2021, 2024)
def incendios_california(fecha):
    if (('2018-08' <= fecha <= '2018-11') or ('2020-08' <= fecha <= '2020-11') or 
        ('2021-08' <= fecha <= '2021-11') or ('2024-08' <= fecha <= '2024-11')):
        return 1
    return 0
df_macro['shock_california_fires'] = df_macro['DATE'].apply(incendios_california)

# 7. Marea Roja en Florida (Cancelaciones 2018)
df_macro['shock_florida_redtide'] = df_macro['DATE'].apply(lambda x: 1 if '2018-07-01' <= x <= '2018-12-31' else 0)

# 8. Huracanes Mayores Florida (Ian Sep-Nov 2022, Milton Oct-Dic 2024)
df_macro['shock_florida_hurricanes'] = df_macro['DATE'].apply(lambda x: 1 if ('2022-09' <= x <= '2022-11') or ('2024-10' <= x <= '2024-12') else 0)

# 9. Crisis Energética/Climática Texas (Apagón invernal 2021 y calores 2023)
df_macro['shock_texas_climate'] = df_macro['DATE'].apply(lambda x: 1 if ('2021-02' in x or '2023-07' <= x <= '2023-09') else 0)


### C. MICRO-EVENTOS COSTA RICA / GUANACASTE

# 10. Huelga Nacional Reforma Fiscal CR (Afectó vías en CR)
df_macro['huelga_nacional_cr_2018'] = df_macro['DATE'].apply(lambda x: 1 if '2018-09-01' <= x <= '2018-11-30' else 0)

# 11. COLAPSO AEROPUERTO LIR Y LLUVIAS (Doble impacto en Guanacaste: Noviembre y Diciembre 2024)
df_macro['colapso_pista_lir_2024'] = df_macro['DATE'].apply(lambda x: 1 if x == '2024-11-30' or x == '2024-11-01' else 0)

print('✅ MATRIZ COMPLETA AÑADIDA: Evaluamos 11 choques en USA, Canadá, Europa y CR.')
display(df_macro.tail(10))

### 3. Siguiente Paso: Integrar Datos de Costa Rica
> **ACCIÓN REQUERIDA:** Busca en la web del ICT el archivo Excel de llegadas internacionales y ocupación. Lo colocas en `data/raw/` para proceder con el Modelado (Fase 4).